# TFT — **Ablation: Wind (Geschwindigkeit vs. Richtung) und Geo, ueber mehrere Seeds**

Diese Auswertung beantwortet: *Tragen **Windgeschwindigkeit** (`WSPM`), **Windrichtung** (`wd`) und
die **Geodaten** (`lon`/`lat`) zur Vorhersage bei — und welche Wind-Variable hilft mehr?*
Dafuer trainieren wir **fuenf Varianten** desselben TFT, alles identisch, nur die Eingaben aendern sich:

| Variante | WSPM | wd | Geo |
|----------|:---:|:---:|:---:|
| **Basis**              | – | – | – |
| **+ Windgeschw.**      | ✓ | – | – |
| **+ Windrichtung**     | – | ✓ | – |
| **+ beide Wind**       | ✓ | ✓ | – |
| **+ beide Wind + Geo** | ✓ | ✓ | ✓ |

**Jede Variante wird ueber mehrere Seeds trainiert.** Wir berichten **Mittelwert ± Standard-
abweichung** des MAE. So sieht man, ob ein Effekt echt ist oder nur Trainingsrauschen.

Daraus lesen wir ab:
- **Windgeschw.-Effekt** = Basis → + Windgeschw.
- **Windrichtungs-Effekt** = Basis → + Windrichtung
- **Welche Wind-Variable ist besser?** → Vergleich + Windgeschw. vs. + Windrichtung
- **Geo-Effekt (zusaetzlich zum Wind)** = + beide Wind → + beide Wind + Geo

> **Einordnung 1:** In allen Varianten kennt das Modell die Station als Kategorie
> (`static_categoricals=["station"]`) und lernt dafuer ein Embedding, das die Lage schon
> **implizit** mitlernen kann. Die Geo-Zeile misst den **zusaetzlichen** Nutzen der *expliziten*
> Koordinaten obendrauf — ein kleiner Wert ist normal.
>
> **Einordnung 2 (fuers Paper):** Das Originalpaper (Zhang & Awang 2025) nutzt Wind nur als
> **Windgeschwindigkeit** (WSPM); **Windrichtung ist dort KEINE Eingabe**. Die `wd`-Varianten
> hier sind eine **eigene Erweiterung** ueber das Paper hinaus.
>
> **Laufzeit & Speicher:** 5 Varianten × Anzahl Seeds. Nach *jedem* Lauf wird der Speicher
> freigegeben (`gc.collect()` + `torch.cuda.empty_cache()`), damit der Kernel bei vielen Laeufen
> nicht vollaeuft. Jeder Lauf zeigt einen **Fortschrittsbalken** und eine `Lauf X/15`-Ueberschrift.

## 1. Bibliotheken

In [1]:
import warnings
warnings.filterwarnings("ignore")

import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

torch.set_float32_matmul_precision("medium")
print("torch:", torch.__version__, "| GPU:", torch.cuda.is_available())

torch: 2.2.2+cu121 | GPU: True


## 2. Daten laden

In [2]:
import io, zipfile, urllib.request, os

DATA_DIR = "beijing_data"
UCI_URL = "https://archive.ics.uci.edu/static/public/501/beijing+multi+site+air+quality+data.zip"
STATIONS = ["Aotizhongxin","Changping","Dingling","Dongsi","Guanyuan","Gucheng",
            "Huairou","Nongzhanguan","Shunyi","Tiantan","Wanliu","Wanshouxigong"]

def try_download():
    try:
        print("Lade UCI-Datensatz ...")
        rawb = urllib.request.urlopen(UCI_URL, timeout=30).read()
        z = zipfile.ZipFile(io.BytesIO(rawb)); os.makedirs(DATA_DIR, exist_ok=True)
        for name in z.namelist():
            if name.endswith(".zip"):
                zipfile.ZipFile(io.BytesIO(z.read(name))).extractall(DATA_DIR)
            elif name.endswith(".csv"):
                z.extract(name, DATA_DIR)
        print("Download OK."); return True
    except Exception as e:
        print("Download fehlgeschlagen:", e); return False

def load_csvs():
    import glob
    files = glob.glob(os.path.join(DATA_DIR, "**", "PRSA_Data_*.csv"), recursive=True)
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if len(files) >= 12 else None

def make_synthetic():
    print(">>> SYNTHETISCHE Demo-Daten <<<")
    rng = np.random.default_rng(0); frames = []
    hours = pd.date_range("2013-03-01", periods=24*60, freq="h")
    for st in STATIONS:
        n=len(hours); daily=30*np.sin(2*np.pi*hours.hour/24)
        frames.append(pd.DataFrame({"year":hours.year,"month":hours.month,"day":hours.day,"hour":hours.hour,
            "PM2.5":(80+daily+rng.normal(0,15,n)).clip(1),"PM10":(100+daily+rng.normal(0,20,n)).clip(1),
            "SO2":rng.normal(15,5,n).clip(0),"NO2":rng.normal(50,15,n).clip(0),"O3":rng.normal(60,20,n).clip(0),
            "TEMP":13+rng.normal(0,2,n),"DEWP":rng.normal(2,5,n),"RAIN":0.0,
            "wd":rng.choice(["N","NE","E","SE","S","SW","W","NW"],n),"WSPM":rng.normal(2,1,n).clip(0),"station":st}))
    return pd.concat(frames, ignore_index=True)

raw = load_csvs() if try_download() else None
if raw is None:
    raw = make_synthetic()
print("Roh-Datensatz:", raw.shape)

Lade UCI-Datensatz ...
Download OK.
Roh-Datensatz: (420768, 18)


## 3. Vorverarbeitung + Koordinaten

In [3]:
df = raw.copy()
df["datetime"] = pd.to_datetime(df[["year","month","day","hour"]])
df = df.sort_values(["station","datetime"]).reset_index(drop=True)
df["time_idx"] = ((df["datetime"] - df["datetime"].min()).dt.total_seconds() // 3600).astype(int)
df["hour"] = df["datetime"].dt.hour
df["dayofweek"] = df["datetime"].dt.dayofweek
df["wd"] = df["wd"].astype(str).replace("nan","unknown").fillna("unknown")

num_cols = [c for c in ["PM2.5","PM10","SO2","NO2","O3","TEMP","DEWP","RAIN","WSPM"] if c in df.columns]
df[num_cols] = (df.groupby("station")[num_cols].apply(lambda g: g.ffill().bfill()).reset_index(level=0, drop=True))
df[num_cols] = df[num_cols].fillna(0)

df = df.rename(columns={"PM2.5": "PM25"})

coord_df = pd.read_csv(os.path.join("..", "data", "stations_geo.csv"))[["station","lon","lat"]]
df = df.merge(coord_df, on="station", how="left")
for c in ["lon","lat"]:
    lo, hi = df[c].min(), df[c].max()
    df[c+"_norm"] = (df[c] - lo) / (hi - lo + 1e-9)
assert df[["lon_norm","lat_norm"]].isna().sum().sum() == 0
print("Vorverarbeitung ok. Stationen:", df["station"].nunique())

Vorverarbeitung ok. Stationen: 12


## 4. Split + Trainings-/Test-Funktion mit Schaltern `use_wspm`, `use_wd`, `use_geo`, `seed`

Am Ende jeder Funktion wird der Speicher freigegeben — wichtig fuer viele Laeufe hintereinander.

In [4]:
max_encoder_length    = 48
max_prediction_length = 12
target = "PM25"

test_days = 14
training_cutoff = df["time_idx"].max() - test_days * 24

# Schadstoffe + Wetter OHNE Wind (Wind wird je nach Schalter ergaenzt)
REALS_BASE = ["PM25", "PM10", "SO2", "NO2", "O3", "TEMP", "DEWP", "RAIN"]

def train_variante(use_wspm, use_wd, use_geo, label, seed, max_epochs=15):
    pl.seed_everything(seed)

    unknown_reals = REALS_BASE + (["WSPM"] if use_wspm else [])
    unknown_cats  = ["wd"] if use_wd else []
    static_reals  = ["lon_norm", "lat_norm"] if use_geo else []

    training = TimeSeriesDataSet(
        df[df["time_idx"] <= training_cutoff],
        time_idx="time_idx", target=target, group_ids=["station"],
        max_encoder_length=max_encoder_length, max_prediction_length=max_prediction_length,
        static_categoricals=["station"],
        static_reals=static_reals,
        time_varying_known_reals=["time_idx", "hour", "dayofweek"],
        time_varying_unknown_reals=unknown_reals,
        time_varying_unknown_categoricals=unknown_cats,
        target_normalizer=GroupNormalizer(groups=["station"]),
        add_relative_time_idx=True, add_target_scales=True,
        add_encoder_length=True, allow_missing_timesteps=True,
    )
    test = TimeSeriesDataSet.from_dataset(training, df,
                                          min_prediction_idx=training_cutoff + 1, stop_randomization=True)

    tl    = training.to_dataloader(train=True,  batch_size=128, num_workers=0)
    testl = test.to_dataloader(train=False, batch_size=256, num_workers=0)

    model = TemporalFusionTransformer.from_dataset(
        training, learning_rate=0.03, hidden_size=32, attention_head_size=2,
        dropout=0.1, hidden_continuous_size=16, loss=QuantileLoss(),
        optimizer="adam", reduce_on_plateau_patience=3,
    )
    ckpt = ModelCheckpoint(monitor="val_loss", mode="min", save_top_k=1)
    trainer = pl.Trainer(
        max_epochs=max_epochs, accelerator="auto", devices=1, gradient_clip_val=0.1,
        callbacks=[EarlyStopping("val_loss", patience=4, mode="min"), ckpt],
        enable_progress_bar=True,      # Fortschrittsbalken AN (man sieht, dass es laeuft)
        enable_model_summary=False,    # grosse Modell-Tabelle AUS (sonst 15x Wiederholung)
        logger=False,
    )
    trainer.fit(model, train_dataloaders=tl, val_dataloaders=testl)
    best = TemporalFusionTransformer.load_from_checkpoint(ckpt.best_model_path) if ckpt.best_model_path else model

    p = best.predict(testl, mode="prediction", return_y=True)
    a = p.output.detach().cpu().numpy().reshape(-1)
    b = p.y[0].detach().cpu().numpy().reshape(-1)
    mae  = float(np.mean(np.abs(a - b)))
    rmse = float(np.sqrt(np.mean((a - b) ** 2)))
    print(f"    -> {label:20s} | seed {seed}: MAE = {mae:.3f} | RMSE = {rmse:.3f}")
    result = {"Variante": label, "seed": seed, "MAE": mae, "RMSE": rmse}

    # --- Speicher freigeben, damit der Kernel bei vielen Laeufen nicht vollaeuft ---
    del model, best, trainer, training, test, tl, testl, p, a, b, ckpt
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

## 5. Alle Varianten ueber mehrere Seeds trainieren

`SEEDS` steuert die Anzahl der Wiederholungen. **3 Seeds** = gute Balance. Bei Zeitnot `[42, 43]`
oder `[42]`. Die `configs`-Liste kannst du ebenfalls kuerzen. Vor jedem Lauf erscheint eine
`Lauf X/15`-Ueberschrift, waehrend des Trainings ein Fortschrittsbalken.

In [ ]:
SEEDS = [42, 43]   # mehr = robuster, aber laenger (5 Varianten x len(SEEDS) Trainings)

configs = [
    dict(label="Basis",              wspm=False, wd=False, geo=False),
    dict(label="+ Windgeschw.",      wspm=True,  wd=False, geo=False),
    dict(label="+ Windrichtung",     wspm=False, wd=True,  geo=False),
    dict(label="+ beide Wind",       wspm=True,  wd=True,  geo=False),
    dict(label="+ beide Wind + Geo", wspm=True,  wd=True,  geo=True),
]

total = len(SEEDS) * len(configs)
alle = []
i = 0
for seed in SEEDS:
    for c in configs:
        i += 1
        print(f"\n########## Lauf {i}/{total}: {c['label']} | seed {seed} — startet ##########")
        alle.append(train_variante(c["wspm"], c["wd"], c["geo"], c["label"], seed=seed))

res_all = pd.DataFrame(alle)   # eine Zeile je (Variante, Seed)
print("\nAlle Einzellaeufe fertig:")
print(res_all.round(3).to_string(index=False))

Seed set to 42



########## Lauf 1/10: Basis | seed 42 — startet ##########


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Seed set to 42


...

Output()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Seed set to 43


    -> + Windrichtung       | seed 43: MAE = 26.487 | RMSE = 46.042

########## Lauf 9/10: + beide Wind | seed 43 — startet ##########


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

## 6. Aggregation ueber Seeds (Mittelwert ± Standardabweichung)

In [1]:
order = [c["label"] for c in configs]
res = (res_all.groupby("Variante")
              .agg(MAE_mean=("MAE", "mean"), MAE_std=("MAE", "std"),
                   RMSE_mean=("RMSE", "mean"), RMSE_std=("RMSE", "std"),
                   n_Seeds=("MAE", "size"))
              .reindex(order).reset_index())

print(f"Ueber {res['n_Seeds'].iloc[0]} Seeds gemittelt (MAE/RMSE in ug/m3):\n")
print(res.round(3).to_string(index=False))

# CSV-Export: Einzellaeufe UND Aggregat
res_all.round(4).to_csv(os.path.join("..", "data", "ergebnis_ablation_alle_seeds.csv"), index=False)
res.round(4).to_csv(os.path.join("..", "data", "ergebnis_ablation_aggregiert.csv"), index=False)
print("\nGespeichert: ergebnis_ablation_alle_seeds.csv und ergebnis_ablation_aggregiert.csv")

NameError: name 'configs' is not defined

## 7. Auswertung: Welche Wind-Variable hilft, und traegt Geo bei?

In [2]:
def m(label):   # Mittelwert-MAE einer Variante
    return float(res.loc[res["Variante"] == label, "MAE_mean"].iloc[0])
def s(label):   # Standardabweichung-MAE
    return float(res.loc[res["Variante"] == label, "MAE_std"].iloc[0])

m_basis, m_ws, m_wr = m("Basis"), m("+ Windgeschw."), m("+ Windrichtung")
m_bw, m_full        = m("+ beide Wind"), m("+ beide Wind + Geo")

def zeile(name, basis, neu):
    delta = basis - neu
    pct = 100 * delta / basis if basis else float("nan")
    print(f"{name:38s}: {delta:+.3f} ug/m3  ({pct:+.2f} %)")

print(f"MAE Basis              : {m_basis:.3f} ± {s('Basis'):.3f}")
print(f"MAE + Windgeschw.      : {m_ws:.3f} ± {s('+ Windgeschw.'):.3f}")
print(f"MAE + Windrichtung     : {m_wr:.3f} ± {s('+ Windrichtung'):.3f}")
print(f"MAE + beide Wind       : {m_bw:.3f} ± {s('+ beide Wind'):.3f}")
print(f"MAE + beide Wind + Geo : {m_full:.3f} ± {s('+ beide Wind + Geo'):.3f}\n")

zeile("Windgeschw.-Effekt (Basis->+WSPM)",   m_basis, m_ws)
zeile("Windrichtungs-Effekt (Basis->+wd)",   m_basis, m_wr)
zeile("Geo-Effekt (+beide Wind->+Geo)",      m_bw,    m_full)

print()
unterschied = abs(m_ws - m_wr)
streuung = max(s("+ Windgeschw."), s("+ Windrichtung"))
besser = "Windgeschwindigkeit" if m_ws < m_wr else "Windrichtung"
print(f"=> Bessere Wind-Variable: {besser} ({min(m_ws,m_wr):.2f} vs {max(m_ws,m_wr):.2f} MAE).")
if unterschied < streuung:
    print("   ABER: Der Unterschied ist kleiner als die Seed-Streuung -> statistisch nicht eindeutig.")
else:
    print("   Der Unterschied ist groesser als die Seed-Streuung -> deutlicher Effekt.")

# Balkendiagramm mit Fehlerbalken (Standardabweichung ueber Seeds)
plt.figure(figsize=(9, 4))
plt.bar(res["Variante"], res["MAE_mean"], yerr=res["MAE_std"], capsize=5,
        color=["#bbb","#69c","#e9a","#7ac","#3b7"])
plt.ylabel("MAE (ug/m3)")
plt.title(f"Ablation ueber {res['n_Seeds'].iloc[0]} Seeds (Fehlerbalken = Std.-Abw.)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout(); plt.show()

NameError: name 'res' is not defined